In [1]:
!pip install nltk rouge-score

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [2]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download('wordnet')
nltk.download('punkt')

def compute_bleu(reference, candidate):
    """
    Compute BLEU score for BLEU-1, BLEU-2, BLEU-3, and BLEU-4 between reference and candidate.
    Uses a smoothing function for short sentences.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    smoothie = SmoothingFunction().method1  # Smoothing for short sequences
    
    bleu_scores = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        bleu_scores[f"BLEU-{n}"] = sentence_bleu(reference_tokens, candidate_tokens, weights=weights, smoothing_function=smoothie)
    
    return bleu_scores

def compute_rouge(reference, candidate):
    """
    Compute ROUGE-L, ROUGE-1, and ROUGE-2 scores.
    Returns the F1 scores.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return {k: v.fmeasure for k, v in scores.items()}

def compute_meteor(reference, candidate):
    """
    Compute METEOR score.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    return meteor_score(reference_tokens, candidate_tokens)




[nltk_data] Downloading package wordnet to /Users/wt/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/wt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
import pandas as pd
import os

results_folder = "results"
result_filename = "simpsons_multi_agent_claude_3_5_haiku_20241022.csv"
evaluation_results_folder = "evaluation_results"

os.makedirs(evaluation_results_folder, exist_ok=True)

result = pd.read_csv(os.path.join(results_folder, result_filename))

for i, row in result.iterrows():

    if i == len(result)-1: # avoid the last row with total and average values
        continue

    if "correct_answer" in row:
        reference_answer = row["correct_answer"].lower()
    elif "truth_answer" in row:
        reference_answer = row["truth_answer"].lower()
    else:
        raise KeyError
        
    generated_answer = row["predicted_answer"].lower()

    bleus = compute_bleu(reference_answer, generated_answer)
    rouges = compute_rouge(reference_answer, generated_answer)
    meteor =  compute_meteor(reference_answer, generated_answer)
    
    for n in range(1, 5):
        result.at[i, f"BLEU-{n}"] = bleus[f"BLEU-{n}"]
    
    for rouge_type, score in rouges.items():
        result.at[i, f"{rouge_type.upper()}"] = score
    
    result.at[i, "METEOR"] = meteor
    

result.to_csv(os.path.join(evaluation_results_folder, result_filename), index=False)


In [4]:
result

,row_num,question_id,question,answer_type,truth_answer,predicted_answer,accuracy,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE1,ROUGE2,ROUGEL,METEOR
0,1,77311,what is on the shelf?,other,book,books,0.750000,0.0,0.000000,0.000000,0.000000,1.0,0.0,1.0,0.5
1,2,12809,how many people are in the picture?,number,1,1,1.000000,1.0,0.316228,0.215443,0.177828,1.0,0.0,1.0,0.5
2,3,1214,are the people sitting or standing?,other,standing,standing,1.000000,1.0,0.316228,0.215443,0.177828,1.0,0.0,1.0,0.5
3,4,88112,what is the group of people doing?,other,standing,standing,1.000000,1.0,0.316228,0.215443,0.177828,1.0,0.0,1.0,0.5
4,5,36705,what are the buildings made of?,other,brick,bricks,0.750000,0.0,0.000000,0.000000,0.000000,1.0,0.0,1.0,0.5
5,6,33098,is there a toy in the picture?,yes/no,yes,yes,1.000000,1.0,0.316228,0.215443,0.177828,1.0,0.0,1.0,0.5
6,7,30161,is there a man on a chair?,yes/no,yes,yes,1.000000,1.0,0.316228,0.215443,0.177828,1.0,0.0,1.0,0.5
7,8,15712,how many people are there?,number,2,2,1.000000,1.0,0.316228,0.215443,0.177828,1.0,0.0,1.0,0.5
8,9,87724,what is the girl doing?,other,standing,posing,0.750000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
9,10,12264,how many people are in the image?,number,1,1,1.000000,1.0,0.316228,0.215443,0.177828,1.0,0.0,1.0,0.5
